# Which attribution method locates the route crossing?

The paper's measurement claim is currently made against one statistic: the
variance share of the retrieval route in the readout. Reviewers have pointed out
that almost nobody uses that quantity as a causal claim, so showing it is biased
risks reading as a strawman.

This notebook runs the comparison against the methods practitioners do use.

**Activation patching** is the ground truth: replace a route's activation with
another trial's and measure the accuracy lost. It is what the paper's `hidden_dep`
and `attn_dep` already are, and it is expensive.

**Attribution patching** is the standard linear approximation to it — the
gradient of a metric with respect to the activation, dotted with the clean-minus-
corrupted difference. One backward pass rather than a forward pass per component.

**Direct logit attribution** is how much each route writes into the correct-class
logit, mean-centred across classes.

**Variance share** is the paper's current proxy and the cheapest of the four.

Each measure has a crossing: the load at which it says the two routes swap
places. Activation patching defines the answer; the question is the *signed*
difference for each of the others, across three task families.

## The two outcomes, and what each means

If only variance share carries a signed offset, the honest finding is that one
cheap statistic is biased while the methods in current use are not — worth
reporting, and it comes with a recommendation.

If attribution patching and direct logit attribution carry the same signed
offset, the finding is about the methods people actually reach for. That is a
much stronger claim and belongs in the abstract.

Either outcome is publishable. The current framing, where the comparison is
against a statistic nobody defends, is the one that is not.

Runtime: about 90 minutes on a T4 at `PRESET = "full"` (270 runs). Run
`PRESET = "smoke"` first — same structure, tiny budgets, roughly eight minutes.
Results resume, and each block syncs a zip to Drive.


In [ ]:
# !pip install torch numpy pandas matplotlib

import copy, hashlib, itertools, json, math, os, pathlib, random, time
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================== CHOOSE HERE ==============================
PRESET  = "full"      # "smoke" (~8 min) | "quick" | "full"
RESUME  = True
WORKDIR = "/content/attr"          # local disk; synced to Drive after each block
DRIVE   = "/content/drive/MyDrive/attr_backup"   # set to None to skip syncing
# =========================================================================
#
# Colab: Runtime > Change runtime type > GPU, and mount Drive first if you want
# the sync:  from google.colab import drive; drive.mount('/content/drive')
#
# Writes go to local disk because Drive buffers them and loses the buffer on
# disconnect. The zip sync after each block is what actually persists.


def S(full, quick=None, smoke=None):
    return {"full": full,
            "quick": quick if quick is not None else max(full // 3, 100),
            "smoke": smoke if smoke is not None else max(full // 25, 100)}[PRESET]


SEEDS = {"full": [0, 1, 2], "quick": [0, 1], "smoke": [0]}[PRESET]
STEPS = S(2500)
ACC_FLOOR = 0.90
TAG = "attr"

WORK = pathlib.Path(WORKDIR)
RESDIR, CKPTDIR = WORK / "results", WORK / "checkpoints"
for _d in (RESDIR, CKPTDIR):
    _d.mkdir(parents=True, exist_ok=True)

print(f"device {DEVICE} | preset {PRESET} | seeds {SEEDS} | {STEPS} steps/run")
print(f"writing to {RESDIR}")
if DEVICE.type == "cpu":
    print("!! CPU — several hours. Switch the runtime to GPU.")


def csv(name):
    return str(RESDIR / f"{name}__{TAG}.csv")


def set_seed(s):
    torch.manual_seed(s); np.random.seed(s); random.seed(s)


def sync():
    if DRIVE:
        import shutil
        try:
            shutil.make_archive(DRIVE, "zip", RESDIR)
        except Exception as e:
            print(f"  (Drive sync skipped: {e})")


In [ ]:
def _jsonable(o):
    if isinstance(o, (np.floating, np.integer)):
        return o.item()
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, torch.Tensor):
        return o.detach().cpu().tolist()
    if isinstance(o, pathlib.Path):
        return str(o)
    return str(o)


def keyof(spec):
    """Content hash of a run specification. Two runs with the same spec are the
    same experiment, so the second one is skipped."""
    return hashlib.md5(
        json.dumps(spec, sort_keys=True, default=_jsonable).encode()).hexdigest()[:16]


def cfg_tag(cfg=None):
    if cfg is None:
        return "default"
    return ",".join(f"{k}={getattr(cfg, k)}" for k in sorted(cfg.__dataclass_fields__))


class Store:
    """Append-only JSONL of finished units of work, one file per block.

    A unit is whatever a block can afford to lose: usually one (condition, seed)
    training run. Records are flushed immediately, so a killed kernel loses at
    most the run in flight. A truncated final line is discarded on load.
    """

    def __init__(self, block):
        self.block = block
        self.path = RESDIR / f"{block}__{TAG}.jsonl"
        self.recs, bad = {}, 0
        if RESUME and self.path.exists():
            for line in self.path.read_text().splitlines():
                if not line.strip():
                    continue
                try:
                    r = json.loads(line)
                    self.recs[r["_key"]] = r
                except (json.JSONDecodeError, KeyError):
                    bad += 1
            if bad:
                print(f"  [{block}] discarded {bad} unreadable record(s)")

    def has(self, spec):
        return RESUME and keyof(spec) in self.recs

    def get(self, spec):
        return self.recs[keyof(spec)]

    def put(self, spec, payload):
        r = {"_key": keyof(spec), "_spec": spec, "_t": time.time(), **payload}
        self.recs[r["_key"]] = r
        with open(self.path, "a") as f:
            f.write(json.dumps(r, default=_jsonable) + "\n")
            f.flush(); os.fsync(f.fileno())
        return r

    def frame(self):
        """Flatten to one row per result. A record either carries metric fields
        directly or a `rows` list, in which case each entry becomes a row."""
        out = []
        for r in self.recs.values():
            spec = {k: v for k, v in r["_spec"].items()}
            body = {k: v for k, v in r.items()
                    if not k.startswith("_") and k not in ("trace", "rows")}
            if "rows" in r:
                for row in r["rows"]:
                    out.append({**spec, **body, **row})
            else:
                out.append({**spec, **body})
        return pd.DataFrame(out)

    def traces(self):
        out = []
        for r in self.recs.values():
            for t in r.get("trace", []):
                out.append({**r["_spec"], **t})
        return pd.DataFrame(out)

    def elapsed(self):
        return sum(r.get("_elapsed", 0.0) for r in self.recs.values())


def cached(store, spec, fn, label=None):
    """Run `fn()` unless this exact spec is already on disk. `fn` returns the
    payload dict; anything not JSON-serialisable must be checkpointed separately."""
    if store.has(spec):
        return store.get(spec)
    t0 = time.time()
    payload = fn()
    payload["_elapsed"] = time.time() - t0
    r = store.put(spec, payload)
    if label:
        print(f"  [{store.block}] {label}  {payload['_elapsed']/60:.1f}m")
    return r


def ckpt_path(name):
    return CKPTDIR / f"{name}__{TAG}.pt"


def ckpt_save(name, obj):
    p = ckpt_path(name)
    torch.save(obj, str(p) + ".tmp")
    os.replace(str(p) + ".tmp", str(p))


def ckpt_load(name):
    p = ckpt_path(name)
    if not (RESUME and p.exists()):
        return None
    try:
        return torch.load(str(p), map_location=DEVICE, weights_only=False)
    except TypeError:
        return torch.load(str(p), map_location=DEVICE)
    except Exception as e:
        print(f"  checkpoint {p.name} unreadable ({e}) — recomputing")
        return None


def save_model(name, m):
    ckpt_save(name, {"state": m.state_dict()})


def load_model_into(name, m):
    ck = ckpt_load(name)
    if ck is None:
        return False
    m.load_state_dict(ck["state"])
    return True


print(f"persistence: {RESDIR}")


## Tasks and model

Three task families and the paper's two-route readout, reproduced here so the
notebook stands alone. `TwoRoute.routes` is the only addition: it returns the two
route activations detached and gradient-enabled, so an attribution can be taken
with respect to each independently.


In [ ]:
PAD, DELAY, CUE, MARK = 0, 1, 2, 3
N_SPECIAL, MAX_POS = 4, 24


@dataclass
class Config:
    n_symbols: int = 8
    max_query_pos: int = MAX_POS
    base_delay: int = 3
    d_model: int = 64
    lr: float = 3e-3
    batch_size: int = 128
    grad_clip: float = 1.0


CFG = Config()
N_CLASSES = CFG.n_symbols
VOCAB = N_SPECIAL + CFG.n_symbols + CFG.max_query_pos


class Task:
    n_out = N_CLASSES

    def __init__(self, cfg=None):
        self.cfg = cfg or CFG

    def _delay(self, B, extra, device):
        return torch.full((B, self.cfg.base_delay + extra), DELAY,
                          device=device, dtype=torch.long)


class SelectiveRecall(Task):
    """`x1..xk . . . Qj` -> report `xj`. The query arrives last, so nothing can
    be stored selectively and both algorithms are viable."""
    name = "selective_recall"

    def generate(self, B, load, extra_delay=0, device=DEVICE):
        c = self.cfg
        lab = torch.randint(0, c.n_symbols, (B, load), device=device)
        j = torch.randint(0, load, (B,), device=device)
        ar = torch.arange(B, device=device)
        q = (j + N_SPECIAL + c.n_symbols).unsqueeze(1)
        return {"x": torch.cat([lab + N_SPECIAL, self._delay(B, extra_delay, device), q], 1),
                "y": lab[ar, j]}


class DelayedRecall(Task):
    """Report the item that appeared most often; no query token."""
    name = "delayed_recall"

    def generate(self, B, load, extra_delay=0, device=DEVICE):
        c = self.cfg
        lab = torch.randint(0, c.n_symbols, (B, load), device=device)
        oh = F.one_hot(lab, c.n_symbols).sum(1)
        cue = torch.full((B, 1), CUE, device=device, dtype=torch.long)
        return {"x": torch.cat([lab + N_SPECIAL, self._delay(B, extra_delay, device), cue], 1),
                "y": oh.argmax(-1)}


class Distractor(Task):
    """A marked target followed by distractors; report the target."""
    name = "distractor"

    def generate(self, B, load, extra_delay=0, device=DEVICE):
        c = self.cfg
        lab = torch.randint(0, c.n_symbols, (B, load), device=device)
        mark = torch.full((B, 1), MARK, device=device, dtype=torch.long)
        cue = torch.full((B, 1), CUE, device=device, dtype=torch.long)
        return {"x": torch.cat([mark, lab + N_SPECIAL,
                                self._delay(B, extra_delay, device), cue], 1),
                "y": lab[:, 0]}


TASKS = {t.name: t for t in [SelectiveRecall, DelayedRecall, Distractor]}
make_task = lambda n: TASKS[n]()


def ablate(v, spec):
    """Resample across the batch: the readout still sees an in-distribution
    vector that carries no information about this trial."""
    if v is None or spec is None:
        return v
    mode, strength = spec if isinstance(spec, (tuple, list)) else (spec, 1.0)
    if strength <= 0:
        return v
    repl = {"shuffle": lambda: v[torch.randperm(v.shape[0], device=v.device)],
            "zero": lambda: torch.zeros_like(v)}[mode]()
    return (1.0 - strength) * v + strength * repl


class TwoRoute(nn.Module):
    """logits = W_h f(h_T) + W_c g(c). Keys and values exclude the final
    timestep, so the retrieval route cannot carry a copy of the recurrent
    state."""

    def __init__(self, cfg=None, n_out=N_CLASSES):
        super().__init__()
        cfg = cfg or CFG
        d = cfg.d_model
        self.d = d
        self.emb = nn.Embedding(VOCAB, d)
        self.rnn = nn.GRU(d, d, batch_first=True)
        self.proj_h = nn.Sequential(nn.Linear(d, d), nn.GELU())
        self.w_h = nn.Linear(d, n_out)
        self.q, self.k, self.v = (nn.Linear(d, d) for _ in range(3))
        self.scale = d ** -0.5
        self.proj_c = nn.Sequential(nn.Linear(d, d), nn.GELU())
        self.w_c = nn.Linear(d, n_out, bias=False)
        self.n_out = n_out

    def routes(self, x):
        """h_T and ctx, detached and gradient-enabled, so an attribution can be
        taken with respect to each route's activation independently."""
        H, _ = self.rnn(self.emb(x))
        H = H.detach()
        h_T = H[:, -1, :].clone().requires_grad_(True)
        Hc = H[:, :-1, :]
        sc = (self.q(h_T).unsqueeze(1) @ self.k(Hc).transpose(1, 2)).squeeze(1) * self.scale
        ctx = torch.einsum("bt,btd->bd", sc.softmax(-1), self.v(Hc))
        ctx = ctx.detach().clone().requires_grad_(True)
        return h_T, ctx

    def readout(self, h_T, ctx):
        return self.w_h(self.proj_h(h_T)), self.w_c(self.proj_c(ctx))

    def forward(self, x, intervene=None):
        iv = intervene or {}
        H, _ = self.rnn(self.emb(x))
        h_T = H[:, -1, :]
        Hc = H[:, :-1, :]
        sc = (self.q(h_T).unsqueeze(1) @ self.k(Hc).transpose(1, 2)).squeeze(1) * self.scale
        ctx = torch.einsum("bt,btd->bd", sc.softmax(-1), self.v(Hc))
        lh = self.w_h(self.proj_h(ablate(h_T, iv.get("hidden"))))
        lc = self.w_c(self.proj_c(ablate(ctx, iv.get("context"))))
        return {"logits": lh + lc, "parts": {"hidden": lh, "context": lc}}


def build(n_out=N_CLASSES):
    return TwoRoute(n_out=n_out)


print(f"vocab {VOCAB}, {N_CLASSES} classes, chance {1/N_CLASSES:.3f}")
print(f"parameters: {sum(p.numel() for p in build().parameters()):,}")
for n in TASKS:
    print(f"  task {n}")


## The four measures


In [ ]:
def train(task_name, load, lam, seed, steps=None):
    steps = steps or STEPS
    task = make_task(task_name)
    set_seed(seed)
    m = build(task.n_out).to(DEVICE)
    opt = torch.optim.AdamW(m.parameters(), lr=CFG.lr)
    for step in range(1, steps + 1):
        cos = 0.5 * (1 + math.cos(math.pi * step / steps))
        for g in opt.param_groups:
            g["lr"] = CFG.lr * cos
        m.train()
        b = task.generate(CFG.batch_size, load)
        o = m(b["x"])
        loss = F.cross_entropy(o["logits"], b["y"])
        if lam > 0:
            loss = loss + lam * o["parts"]["context"].norm(dim=-1).mean()
        opt.zero_grad(set_to_none=True); loss.backward()
        nn.utils.clip_grad_norm_(m.parameters(), CFG.grad_clip); opt.step()
    return m, task


@torch.no_grad()
def accuracy(m, task, load, intervene=None, n=8):
    m.eval(); c = t = 0
    for _ in range(n):
        b = task.generate(CFG.batch_size, load)
        c += (m(b["x"], intervene=intervene)["logits"].argmax(-1) == b["y"]).sum().item()
        t += b["y"].numel()
    m.train(); return c / t


# ---------------------------------------------------------------------------
# Four ways of asking "which route matters", in decreasing order of cost.
#
# 1. ACTIVATION PATCHING (the ground truth here). Replace a route's activation
#    with another trial's and measure the accuracy lost. Expensive — a forward
#    pass per route per batch — and the quantity every other measure is
#    approximating.
# 2. ATTRIBUTION PATCHING. The standard linear approximation to (1): the
#    gradient of a metric with respect to the activation, dotted with the
#    difference between the clean and corrupted activation. One backward pass
#    for all routes at once.
# 3. DIRECT LOGIT ATTRIBUTION. How much each route writes into the correct-class
#    logit, mean-centred across classes.
# 4. VARIANCE SHARE. The cheapest proxy: the fraction of readout standard
#    deviation contributed by the retrieval route.
#
# The paper's claim is about where each of these locates the crossing between
# routes. (1) defines the answer; the question is whether (2)-(4) agree, and if
# not, whether they disagree in a consistent direction.
# ---------------------------------------------------------------------------

@torch.no_grad()
def activation_patching(m, task, load, n=8, mode="shuffle"):
    acc = accuracy(m, task, load, n=n)
    den = max(acc - 1.0 / m.n_out, 1e-6)
    d = lambda a: float(np.clip((acc - a) / den, 0, 1))
    return {"acc_clean": acc,
            "hidden_dep": d(accuracy(m, task, load, {"hidden": (mode, 1.0)}, n)),
            "attn_dep": d(accuracy(m, task, load, {"context": (mode, 1.0)}, n))}


def attribution_patching(m, task, load, n=8):
    """Syed et al.'s linear approximation: attr = (a_clean - a_corrupt) . dM/da,
    with M the correct-class logit, mean-centred. The corruption is the same
    resampling activation patching uses, so the two measure the same
    intervention — one exactly, one to first order."""
    m.eval()
    ah, ac = [], []
    for _ in range(n):
        b = task.generate(CFG.batch_size, load)
        h_T, ctx = m.routes(b["x"])
        lh, lc = m.readout(h_T, ctx)
        lg = lh + lc
        M = (lg.gather(1, b["y"][:, None]).squeeze(1) - lg.mean(-1)).mean()
        gh, gc = torch.autograd.grad(M, [h_T, ctx])
        perm = torch.randperm(h_T.shape[0], device=h_T.device)
        ah.append(float(((h_T - h_T[perm]) * gh).sum(-1).abs().mean()))
        ac.append(float(((ctx - ctx[perm]) * gc).sum(-1).abs().mean()))
    m.train()
    a_h, a_c = float(np.mean(ah)), float(np.mean(ac))
    return {"attr_hidden": a_h, "attr_context": a_c,
            "attr_share": a_c / (a_h + a_c + 1e-9)}


@torch.no_grad()
def direct_logit_attribution(m, task, load, n=8):
    """Each route's write into the correct-class logit, mean-centred across
    classes. Negative parts are clipped at zero before forming the share: a
    route writing away from the answer is not evidence that the other route is
    doing more of the work."""
    m.eval(); dh, dc = [], []
    for _ in range(n):
        b = task.generate(CFG.batch_size, load)
        o = m(b["x"])
        for part, acc_ in [("hidden", dh), ("context", dc)]:
            L = o["parts"][part]
            acc_.append(float((L.gather(1, b["y"][:, None]).squeeze(1)
                               - L.mean(-1)).mean()))
    m.train()
    a, c = max(float(np.mean(dh)), 0.0), max(float(np.mean(dc)), 0.0)
    return {"dla_hidden": float(np.mean(dh)), "dla_context": float(np.mean(dc)),
            "dla_share": c / (a + c + 1e-9)}


@torch.no_grad()
def variance_share(m, task, load, n=4):
    m.eval(); sh, sc = [], []
    for _ in range(n):
        o = m(task.generate(CFG.batch_size, load)["x"])
        sh.append(o["parts"]["hidden"].std(-1)); sc.append(o["parts"]["context"].std(-1))
    m.train()
    h, c = float(torch.cat(sh).mean()), float(torch.cat(sc).mean())
    return {"context_share": c / (h + c + 1e-9)}


def all_measures(m, task, load):
    out = {}
    out.update(activation_patching(m, task, load))
    out.update(attribution_patching(m, task, load))
    out.update(direct_logit_attribution(m, task, load))
    out.update(variance_share(m, task, load))
    return out


def crossing(x, a, b):
    """Load at which two curves cross, by linear interpolation."""
    x = np.asarray(x, float); d = np.asarray(a, float) - np.asarray(b, float)
    ok = np.isfinite(d); x, d = x[ok], d[ok]
    o = np.argsort(x); x, d = x[o], d[o]
    for i in range(len(d) - 1):
        if d[i] == 0:
            return float(x[i])
        if d[i] * d[i + 1] < 0:
            f = abs(d[i]) / (abs(d[i]) + abs(d[i + 1]))
            return float(x[i] + f * (x[i + 1] - x[i]))
    return np.nan


def share_crossing(x, s, thr=0.5):
    return crossing(x, np.asarray(s, float), np.full(len(s), thr))


def boot_ci(vals, n=10000, alpha=0.05, seed=0):
    v = np.asarray([x for x in vals if x == x], float)
    if len(v) < 2:
        return (float(v.mean()) if len(v) else np.nan, np.nan, np.nan)
    rng = np.random.default_rng(seed)
    mu = rng.choice(v, size=(n, len(v)), replace=True).mean(1)
    return float(v.mean()), float(np.quantile(mu, alpha/2)), float(np.quantile(mu, 1-alpha/2))


## The comparison


In [ ]:
FAMILIES = ["selective_recall", "delayed_recall", "distractor"]
LAMS  = [0.0, 0.005, 0.02, 0.05, 0.1]
LOADS = [2, 4, 6, 8, 12, 16]
if PRESET != "full":
    FAMILIES, LAMS, LOADS = FAMILIES[:2], LAMS[:3], LOADS[::2]

st = Store("measures")
units = list(itertools.product(FAMILIES, LAMS, LOADS, SEEDS))
todo = [u for u in units
        if not st.has({"block": "measures", "task": u[0], "lam": u[1],
                       "load": u[2], "seed": u[3], "steps": STEPS})]
print(f"{len(units)} units, {len(units)-len(todo)} cached, {len(todo)} to run "
      f"(~{len(todo)*STEPS/1e3:.0f}k optimiser steps)\n")

t0 = 0
for i, (tk, lam, load, sd) in enumerate(todo):
    spec = {"block": "measures", "task": tk, "lam": lam, "load": load,
            "seed": sd, "steps": STEPS}
    cached(st, spec,
           lambda tk=tk, lam=lam, load=load, sd=sd: (
               lambda mt: all_measures(mt[0], mt[1], load))(train(tk, load, lam, sd)))
    if (i + 1) % 20 == 0:
        print(f"  {i+1}/{len(todo)}"); sync()

M = st.frame(); M.to_csv(csv("m1_measures"), index=False)
sync()
print(f"\n{len(M)} rows written")

# ---- where does each measure put the crossing? ----------------------------
# Activation patching defines the answer: the load at which the two routes are
# equally necessary. Each cheaper measure has its own crossing — the load at
# which its share passes one half — and the question is the signed difference.
rows = []
for (tk, lam, sd), g in M.groupby(["task", "lam", "seed"]):
    if lam == 0 or g.load.nunique() < 3:
        continue
    g = g.sort_values("load")
    if g.acc_clean.min() < ACC_FLOOR:
        g = g[g.acc_clean >= ACC_FLOOR]
        if g.load.nunique() < 3:
            continue
    ap = crossing(g.load, g.hidden_dep, g.attn_dep)
    if not np.isfinite(ap):
        continue
    rows.append({"task": tk, "lam": lam, "seed": sd, "activation_patching": ap,
                 "attribution_patching": share_crossing(g.load, g.attr_share),
                 "direct_logit_attr": share_crossing(g.load, g.dla_share),
                 "variance_share": share_crossing(g.load, g.context_share)})
C = pd.DataFrame(rows); C.to_csv(csv("m2_crossings"), index=False)

PROXIES = ["attribution_patching", "direct_logit_attr", "variance_share"]
for p in PROXIES:
    C[f"offset_{p}"] = C[p] - C.activation_patching

print("\ncrossing location by measure (loads), mean over seeds:")
print(C.groupby(["task", "lam"])[["activation_patching"] + PROXIES]
      .mean().round(2).to_string())

print("\n" + "=" * 78)
print("offset from activation patching, by proxy")
print("  negative = the proxy places the crossing at a LOWER load than the")
print("  ablation does, i.e. it reports the handover as happening earlier\n")
summ = []
for p in PROXIES:
    fam = C.groupby("task")[f"offset_{p}"].mean().dropna()
    if not len(fam):
        continue
    mu, lo, hi = boot_ci(fam.values)
    k, n = int((fam < 0).sum()), len(fam)
    summ.append({"proxy": p, "families": n, "negative": k,
                 "mean offset": mu, "lo": lo, "hi": hi,
                 "sign test p": 2 * 0.5 ** n if k in (0, n) else np.nan,
                 "per family": ", ".join(f"{v:+.2f}" for v in fam.values)})
    print(f"  {p:<22} {k}/{n} families negative   "
          f"mean {mu:+.2f} loads" + (f"  95% CI [{lo:+.2f}, {hi:+.2f}]"
                                      if np.isfinite(lo) else "") +
          (f"   sign test p = {2*0.5**n:.4f}" if k in (0, n) else ""))
    print(f"  {'':<22} per family: {summ[-1]['per family']}")
SM = pd.DataFrame(summ); SM.to_csv(csv("m3_summary"), index=False)
print("=" * 78)

print("\n  What this decides. If the offset is signed and consistent only for")
print("  variance share, the finding is that one cheap statistic is biased and")
print("  the methods practitioners actually use are not — report it that way and")
print("  recommend attribution patching. If attribution patching and direct")
print("  logit attribution carry the same signed offset, the finding is about")
print("  the methods in current use, which is a considerably stronger claim and")
print("  belongs in the abstract.")
sync()
